# Faruq-v3 — selected-DCT FFAB2 efficiency stage
Run only after the fair from-start Stage-1 decision is PASS. One seed per runtime. Test remains locked.

In [ ]:
SEED=42  # use 123 or 2026 on other runtimes
assert SEED in (42,123,2026)
BRANCH='codex/af2-ffab2-from-start-dct'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import importlib,json,os,shutil,subprocess,sys,tarfile,time,torch
from pathlib import Path
assert torch.cuda.is_available(),'Aktifkan T4 GPU.'
REPO=Path('/content/coffee-bean-detection'); os.chdir('/content')
if REPO.exists(): shutil.rmtree(REPO)
clone=['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)]
for attempt in range(1,4):
    result=subprocess.run(clone)
    if result.returncode==0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt==3: raise RuntimeError('Git clone gagal tiga kali.')
    time.sleep(2)
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96','-e',str(REPO)],check=True)
for name in list(sys.modules):
    if name=='coffee_detector' or name.startswith('coffee_detector.'): sys.modules.pop(name,None)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
print('GPU:',torch.cuda.get_device_name(0),'| SEED:',SEED)

In [ ]:
from coffee_detector.drive_project import require_project_artifact,resolve_drive_project_root
D0_REL=f'experiments/faruq-v3-acmc-paired-confirmation-v1/D0_base/D0_seed{SEED}/weights/best.pt'
STAGE1_REL='experiments/faruq-v3-af2-ffab2-from-start-v1/val_reports/af2_ffab2_from_start_decision.json'
REQ=('bundles/faruq-development-v3-grouped.tar',D0_REL,STAGE1_REL)
PROJECT=resolve_drive_project_root(required_relative_paths=REQ)
ARCHIVE=require_project_artifact(PROJECT,REQ[0]); D0=require_project_artifact(PROJECT,D0_REL); STAGE1=require_project_artifact(PROJECT,STAGE1_REL)
decision=json.loads(STAGE1.read_text()); assert decision['decision']=='PASS' and decision['next']=='AUTHORIZE_DCT_EFFICIENCY_STAGE' and decision['test_opened'] is False
DATA=Path('/content/faruq-development-v3-grouped')
if not (DATA/'faruq_grouped_summary.json').is_file():
    if DATA.exists(): shutil.rmtree(DATA)
    with tarfile.open(ARCHIVE,'r') as archive: archive.extractall('/content',filter='data')
assert (DATA/'data.yaml').is_file() and not (DATA/'test').exists()
OUTPUT=PROJECT/'experiments/faruq-v3-af2-ffab2-from-start-v1'
STATIC=OUTPUT/'val_reports'/f'from_start_static_audit_seed{SEED}.json'
print('STAGE1 PASS:',STAGE1); print('D0:',D0)

In [ ]:
from coffee_detector.af2_ffa import run_af2_ffa_from_start_static_audit
audit=run_af2_ffa_from_start_static_audit(D0,STATIC,device='cuda:0')
print('STATIC:',audit['decision']); print('DCT:',audit['records']['AF2FFADCTFS'])
assert audit['decision']=='PASS','STOP: static audit gagal.'

In [ ]:
ARM='AF2FFADCTFS'; RESULT=OUTPUT/'val_reports'/f'{ARM}_seed{SEED}_result.json'; LOG=OUTPUT/f'{ARM}_seed{SEED}_run.log'
if RESULT.is_file():
    print('REUSE COMPLETE:',RESULT)
else:
    command=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_af2_ffa_from_start_arm','--arm',ARM,'--data-root',str(DATA),'--grouped-summary',str(DATA/'faruq_grouped_summary.json'),'--d0-checkpoint',str(D0),'--static-audit',str(STATIC),'--stage1-decision',str(STAGE1),'--output-root',str(OUTPUT),'--seed',str(SEED),'--device','0','--authorize-training']
    print('START/RESUME:',ARM,'seed',SEED,'| log=',LOG,flush=True)
    with LOG.open('a',encoding='utf-8') as stream: process=subprocess.Popen(command,cwd=REPO,stdout=stream,stderr=subprocess.STDOUT,text=True)
    seen=None
    while process.poll() is None:
        csv=OUTPUT/ARM/f'{ARM}_seed{SEED}'/'results.csv'; epochs=max(0,len(csv.read_text(errors='replace').splitlines())-1) if csv.is_file() else 0
        if epochs!=seen: print(f'{ARM} seed {SEED}: {epochs}/50 epoch tercatat',flush=True); seen=epochs
        time.sleep(60)
    if process.returncode:
        print('\n'.join(LOG.read_text(errors='replace').splitlines()[-150:])); raise RuntimeError(f'{ARM} gagal: {process.returncode}')
payload=json.loads(RESULT.read_text()); print({k:payload['metrics'][k] for k in ('macro_map50_95','bottom3_class_map50_95','worst_class_map50_95')})

In [ ]:
R=[OUTPUT/'val_reports'/f'AF2FFAB2FS_seed{s}_result.json' for s in (42,123,2026)]
D=[OUTPUT/'val_reports'/f'AF2FFADCTFS_seed{s}_result.json' for s in (42,123,2026)]
if all(path.is_file() for path in R+D):
    SUMMARY=OUTPUT/'val_reports/af2_ffab2_dct_efficiency_decision.json'
    cmd=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_af2_ffa_dct_decision','--rfft',*[str(p) for p in R],'--dct',*[str(p) for p in D],'--output',str(SUMMARY),'--device','0']
    subprocess.run(cmd,cwd=REPO,check=True)
    final=json.loads(SUMMARY.read_text()); print('DCT DECISION:',final['decision']); print('NEXT:',final['next']); print('EFFICIENCY:',final['efficiency'])
else:
    print('MENUNGGU DCT SEED LAIN:',{str(p):p.is_file() for p in D})
print('Jangan membuka test.')